# Recipe 02 — Built-in Steps
> Explore the ready-made pipeline steps that ship with Anchor.

| | |
|---|---|
| **Module** | `anchor.pipeline.steps` |
| **Key classes** | `retriever_step`, `filter_step`, `reranker_step`, `query_transform_step` |
| **Difficulty** | Beginner |

In [ ]:
from anchor.pipeline import ContextPipeline
from anchor.pipeline.steps import (
    retriever_step,
    filter_step,
    postprocessor_step,
    reranker_step,
    auto_promotion_step,
    graph_retrieval_step,
    classified_retriever_step,
    query_transform_step,
)
from anchor.models import QueryBundle, ContextItem, SourceType

## 1 — Mock helpers
We create lightweight mocks so every cell runs without external services.

In [ ]:
# --- mock retriever ---
class MockRetriever:
    def retrieve(self, query, top_k=5):
        return [
            ContextItem(
                id=f"doc-{i}",
                content=f"Document {i} about {query.query_str}",
                source=SourceType.RETRIEVAL,
                score=round(0.95 - i * 0.1, 2),
                token_count=15,
            )
            for i in range(top_k)
        ]

# --- mock reranker ---
class MockReranker:
    def rerank(self, items, query, top_k=3):
        scored = sorted(items, key=lambda x: x.score or 0, reverse=True)
        return scored[:top_k]

retriever = MockRetriever()
reranker  = MockReranker()
print("Mocks ready")

## 2 — `retriever_step`
Wraps any retriever into a pipeline step. The step calls
`retriever.retrieve(query, top_k=...)` and feeds items downstream.

In [ ]:
step = retriever_step(name="retriever", retriever=retriever, top_k=10)
print(f"Step name: {step.name}")

## 3 — `filter_step`
Accepts a predicate function `(items, query) -> items` to prune results.

In [ ]:
step = filter_step(
    name="quality_filter",
    filter_fn=lambda items, q: [i for i in items if (i.score or 0) > 0.5],
)
print(f"Step name: {step.name}")

## 4 — `reranker_step`
Re-scores items with a reranker model and keeps only `top_k`.

In [ ]:
step = reranker_step(name="rerank", reranker=reranker, top_k=5)
print(f"Step name: {step.name}")

## 5 — `query_transform_step`
Rewrites or expands the query before retrieval.

In [ ]:
class MockTransformer:
    def transform(self, query):
        return QueryBundle(query_str=f"expanded: {query.query_str}")

step = query_transform_step(name="expand", transformer=MockTransformer())
print(f"Step name: {step.name}")

## 6 — Assemble a full pipeline
Chain multiple built-in steps into a single pipeline.

In [ ]:
from anchor.formatters import GenericTextFormatter

pipeline = ContextPipeline(max_tokens=4096)
pipeline.add_system_prompt("You are a helpful assistant.")
pipeline.with_formatter(GenericTextFormatter())

pipeline.add_step(retriever_step("docs", retriever=retriever, top_k=10))
pipeline.add_step(filter_step(
    "quality",
    filter_fn=lambda items, q: [i for i in items if (i.score or 0) > 0.5],
))
pipeline.add_step(reranker_step("rerank", reranker=reranker, top_k=5))

query = QueryBundle(query_str="context engineering best practices")
result = pipeline.build(query)

print(f"Items included : {len(result.window.items)}")
print(f"Token usage    : {result.window.used_tokens}")

## Key Takeaways
- Built-in step factories (`retriever_step`, `filter_step`, etc.) cover the most common patterns.
- Steps compose left-to-right: each step receives items from the previous one.
- Use mock objects during development to iterate quickly.

**Next:** [Custom Steps](03_custom_steps.ipynb)